<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_01_cnn_image_classification_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.  Ch. 10 for recurrence and the LSTM.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_05 · Notebook 01 — A Convolution, by Hand and Then Learned

**Deep Learning for Engineering · Aalborg University · Part 1**

L5.1 opens with one sentence that carries the whole lecture:

> **A learned local rule, applied everywhere.**

This notebook is the first half of that sentence made concrete on a grid. You
will write a convolution in NumPy, watch three hand-chosen kernels respond to
three kinds of weld defect, count the parameters a convolution saves against a
dense layer, and then train a small convolutional network to sort radiographs
into *clean*, *crack* and *pit*.

The task is the weld-inspection example from **L3.2**. A radiographer looks at a
film of a welded seam and decides whether it is sound, cracked, or porous. The
images here are drawn procedurally rather than measured — twenty lines of NumPy,
two Gaussian profiles and some film grain — and no radiographer would be fooled
by them. What is kept is the property the architecture is about:

**the defect can be anywhere on the plate, and it is the same defect wherever it
is.**

That property has a name, and the name matters because L5.1 uses it precisely.
A function is **equivariant** to translation if moving the input moves the output
the same way; it is **invariant** if moving the input does not change the output
at all. A convolution layer is equivariant: shift the crack three pixels right
and the feature map shifts three pixels right. The classifier on top is meant to
be invariant: shift the crack three pixels right and it is still a crack. You get
from one to the other by pooling, which is exactly what pooling is for.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_5_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex05-cnn-and-gnn/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_5_core as core

core.set_seed(0)

X, y = core.weld_images(n_per_class=200, seed=3)
X_train, y_train, X_test, y_test = core.split_data(X, y, frac=0.7, seed=5)

print("all      :", X.shape, " class counts", np.bincount(y))
print("training :", X_train.shape)
print("held out :", X_test.shape)

core.plot_images(X_train, y_train, n=12, title="Twelve of the training images")
plt.show()

**What you should see.** `all : (600, 1, 16, 16)` with class counts
`[200 200 200]`, 420 training images, 180 held out, and a grid of twelve
labelled radiographs.

Six hundred images is a small dataset by the standards of the vision literature
and a perfectly normal one by the standards of engineering inspection, where
every labelled example costs a technician's time. Keep that number in mind: it is
the reason the network at the end of this notebook has two thousand parameters
and not two million.

---

## 1 · A convolution, written out

A two-dimensional convolution is the equation on L5.1's slide:

$$h_{ij} = a\!\left(\sum_m \sum_n k_{mn}\, x_{i+m,\,j+n} + b\right)$$

In words: put a small window on the image at position $(i, j)$, multiply the
window by the kernel element by element, add up the result, add a bias, and pass
it through an activation. Then slide the window by one pixel and do it again.
The kernel $k$ is the **same** at every position — that is the "applied
everywhere" half of the sentence, and it is why it is called *weight sharing*.

Two details that trip people up.

**This is not, strictly, a convolution.** A mathematician's convolution flips the
kernel before sliding it. The equation above does not flip, which makes it a
*cross-correlation*. PyTorch's `nn.Conv2d` does not flip either. Nobody minds,
because the kernel is learned: if a flipped kernel were better, the network would
learn the flipped one. It matters only when you compare against a textbook or
against `scipy.signal.convolve2d`, and then it matters a great deal.

**The output is smaller than the input.** A 3 by 3 kernel on a 5 by 5 image can
only be centred where the whole window fits, so the output is 3 by 3. This is
called *valid* convolution. If you want the output to stay 5 by 5 you must pad
the input with a border of zeros — `padding=1` in PyTorch — and then you are
inventing pixels outside the plate, which is a modelling assumption rather than a
formatting choice.

### Your turn

Write `conv2d(image, kernel)` for a single greyscale image and a single kernel,
with no padding and a stride of one. Two nested loops are fine; this is about
understanding, not speed.

A correct answer applied to `core.TEST_PATCH` — a 5 by 5 patch with a bright
vertical bar down the middle column — with `core.KERNELS["vertical edge"]`
returns a 3 by 3 array whose columns are `4`, `0` and `-4`.

In [ ]:
# TODO 1 --- a 2-D convolution by hand ----------------------------------------------
# One `...` to replace, inside the two loops:
#   np.sum(image[i:i + kh, j:j + kw] * kernel)     the patch under the kernel, times it, summed
def conv2d(image, kernel):
    """Cross-correlation of `image` with `kernel`: no padding, stride 1, no bias."""
    h, w   = image.shape
    kh, kw = kernel.shape
    out    = np.zeros((h - kh + 1, w - kw + 1))
    for i in range(h - kh + 1):
        for j in range(w - kw + 1):
            out[i, j] = ...                       # <- np.sum(image[i:i + kh, j:j + kw] * kernel)
    return out
# ------------------------------------------------------------------------------

In [ ]:
k = core.KERNELS["vertical edge"]
result = conv2d(core.TEST_PATCH, k)

print("patch:")
print(core.TEST_PATCH.astype(int))
print("\nkernel:")
print(k.astype(int))
print("\nresult:")
print(np.round(result, 6))
print("\nshape", core.TEST_PATCH.shape, "->", result.shape)

**What you should see.**

```
result:
[[ 4.  0. -4.]
 [ 4.  0. -4.]
 [ 4.  0. -4.]]

shape (5, 5) -> (3, 3)
```

Read those nine numbers rather than glancing at them, because they say exactly
what an edge detector does.

The left column is `+4`: there, the bright bar sits under the kernel's positive
right-hand side, so the window straddles a dark-to-bright transition. The right
column is `-4`: the bar is now under the negative left-hand side, a
bright-to-dark transition. The middle column is `0`: the bar is under the
kernel's zero centre column, so a *symmetric* feature produces no response at
all.

An edge detector responds to change and ignores uniformity. A dense layer would
have had to learn that from data. Here it is three numbers wide, and it took no
data at all.

---

## 2 · What the kernels see

`core.KERNELS` holds four fixed 3 by 3 kernels: a vertical edge detector, a
horizontal one, a centre-surround "blob" detector, and a blur. Apply them to one
example of each class and look at what comes out.

Predict, before you run it: which kernel should shout loudest at a crack, and
which at a pit?

In [ ]:
# TODO 2 --- four kernels against three classes ------------------------------------------
# Two `...` to replace, inside the loops:
#   line 1  ->  X_train[y_train == c][0, 0]         the first training image of class c, 16x16
#   line 2  ->  conv2d(img, kernel)                  its response to this kernel
responses, peak, examples = {}, {}, {}
for c, cname in enumerate(core.CLASS_NAMES):
    img = ...                                     # <- X_train[y_train == c][0, 0]
    examples[cname] = img
    for name, kernel in core.KERNELS.items():
        r = ...                                   # <- conv2d(img, kernel)
        responses[(cname, name)] = r
        peak[(cname, name)] = float(np.abs(r).max())
# ------------------------------------------------------------------------------

In [ ]:
names = list(core.KERNELS)
print("peak absolute response")
print("            " + "".join(f"{n:>18s}" for n in names))
for cname in core.CLASS_NAMES:
    print(f"  {cname:9s} " + "".join(f"{peak[(cname, n)]:18.3f}" for n in names))

fig, axes = plt.subplots(3, len(names) + 1, figsize=(2.0 * (len(names) + 1), 6.0))
for r, cname in enumerate(core.CLASS_NAMES):
    axes[r, 0].imshow(examples[cname], cmap="gray", vmin=0, vmax=1)
    axes[r, 0].set_ylabel(cname, fontsize=10)
    axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
    if r == 0:
        axes[r, 0].set_title("image", fontsize=9)
    for c, n in enumerate(names):
        axes[r, c + 1].imshow(responses[(cname, n)], cmap="RdBu_r")
        axes[r, c + 1].axis("off")
        if r == 0:
            axes[r, c + 1].set_title(n, fontsize=9)
fig.suptitle("Four fixed kernels against three defect classes", fontsize=12)
fig.tight_layout()
plt.show()

**What you should see.** A three-by-five grid of pictures and a table of peak
responses. The exact numbers depend on which images the split happened to put
first, but the pattern is robust:

* the **blob** kernel gives its largest response on the *pit*, because a pit is
  precisely a centre that differs from its surround;
* the two **edge** kernels respond strongly to the *crack*, and which of the two
  wins depends on the angle of that particular crack;
* the **blur** kernel responds to everything and distinguishes nothing — its
  output is a smoothed copy of the input.

Two lessons, and the second is the important one.

**Different local rules pick out different structure.** Nothing here was learned.
A century of image processing consisted of engineers choosing kernels like these
by hand, and it worked well when the thing being detected was simple enough to
describe.

**A hand-chosen kernel bank does not scale.** You could add a diagonal edge
detector, and one for a differently sized pit, and one for a crack that is
brighter rather than darker. Each is another judgement call, and you will run out
of patience long before you run out of defect types. A convolutional network
keeps the structure — small kernels, applied everywhere — and lets the *data*
choose the numbers inside them. That is the entire idea.

---

## 3 · The parameter argument

L5.1 puts two counts side by side:

$$P_{\mathrm{dense}} = N_{\mathrm{in}} N_{\mathrm{out}}
\qquad
P_{\mathrm{conv}} = K^2 C_{\mathrm{in}} C_{\mathrm{out}}$$

The point is that the convolution count has **no image size in it**. Doubling
the resolution quadruples $N_{\mathrm{in}}$ and $N_{\mathrm{out}}$ and so
multiplies the dense count by sixteen, while the convolution count does not move
at all.

### Your turn

Compute both counts for a layer that takes a $16 \times 16$ single-channel image
and produces **eight** feature maps of the same size, and then for the same
layer on a $256 \times 256$ image. Use a $3 \times 3$ kernel. Include biases:
a dense layer has one bias per output unit, a convolution has one per output
*channel*.

In [ ]:
# TODO 3 --- four parameter counts -----------------------------------------------------
# Two `...` to replace (the 16 x 16 pair is written; write the 256 x 256 pair the same way):
#   dense_256  ->  256*256 * (256*256*8) + (256*256*8)
#   conv_256   ->  3*3 * 1 * 8 + 8              the same 80: a kernel does not know the image size
dense_16  = 16*16 * (16*16*8) + (16*16*8)        # every input pixel to every output
conv_16   = 3*3 * 1 * 8 + 8                      # eight 3x3 kernels and eight biases
dense_256 = ...                                   # <- 256*256 * (256*256*8) + (256*256*8)
conv_256  = ...                                   # <- 3*3 * 1 * 8 + 8
assert not any(v is ... for v in (dense_256, conv_256)), "TODO 3: replace the two ..."
# ------------------------------------------------------------------------------

In [ ]:
rows = [["16 x 16", f"{dense_16:,}", f"{conv_16:,}",
         f"{dense_16 / conv_16:,.0f} x"],
        ["256 x 256", f"{dense_256:,}", f"{conv_256:,}",
         f"{dense_256 / conv_256:,.0f} x"]]
print(core.error_table(rows, ["image", "dense layer", "conv layer", "ratio"]))

conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)
print("\nPyTorch agrees:", core.count_parameters(conv), "parameters in nn.Conv2d(1, 8, 3)")

**What you should see.**

| image | dense layer | conv layer | ratio |
| --- | --- | --- | --- |
| 16 x 16 | 526,336 | 80 | 6,579 x |
| 256 x 256 | 34,360,262,656 | 80 | 429,503,283 x |

and `PyTorch agrees: 80 parameters`.

Thirty-four **billion** parameters for one layer of a network looking at a
modest 256 by 256 image. That number is not a rhetorical flourish; it is why
nobody trained a useful dense network on images, and why the field was stuck
until weight sharing was taken seriously.

Note what the eighty parameters buy and what they do not. The convolution can
represent any bank of eight $3 \times 3$ local rules. It **cannot** represent
"pixel (3, 7) matters more than pixel (11, 2)", because it has no way to say
where it is. If the defect always appeared in the same corner of the plate, the
dense layer's extra parameters would be earning their keep, and a convolution
would be the wrong choice. Architecture is an assumption about the problem, and
this one is the assumption that position does not matter.

---

## 4 · The network

Now build the thing. A conventional small convolutional network alternates
convolution, activation and pooling, then flattens and classifies:

```
input   (1, 16, 16)
Conv2d(1 -> 8, 3x3, padding=1)   ->  (8, 16, 16)
ReLU
MaxPool2d(2)                     ->  (8, 8, 8)
Conv2d(8 -> 16, 3x3, padding=1)  ->  (16, 8, 8)
ReLU
MaxPool2d(2)                     ->  (16, 4, 4)
Flatten                          ->  (256,)
Linear(256 -> 3)                 ->  (3,)
```

Three ideas are doing the work.

**Padding of one** keeps the spatial size unchanged through the convolution, so
the only thing that shrinks the image is the pooling. That makes the arithmetic
easy to follow, which matters more here than the tiny artefact introduced at the
border.

**Pooling** takes the maximum over each 2 by 2 block. It halves the resolution,
which is what lets the second layer's 3 by 3 kernel cover a 6 by 6 patch of the
original image — the *receptive field* growing with depth. It also throws away
exactly where within the block the maximum was, which is the step that turns
equivariance into invariance.

**Three outputs, no softmax.** `nn.CrossEntropyLoss` applies the softmax itself,
in a numerically stabler way than doing it in two steps. Add your own softmax
before the loss and you will apply it twice, which trains, slowly, to a worse
answer, and prints no warning.

### Your turn

Build that network with `nn.Sequential` and count its parameters.

In [ ]:
# TODO 4 --- the convolutional network -----------------------------------------------------
# Three `...` to replace, in order:
#   line 1  ->  nn.Conv2d(8, 16, kernel_size=3, padding=1)     second conv block: 8 -> 16 channels
#   line 2  ->  nn.MaxPool2d(2)                                 16x16 -> 8x8 -> 4x4 after two pools
#   line 3  ->  nn.Linear(16 * 4 * 4, 3)                        16 channels of 4x4, three classes
core.set_seed(0)
cnn = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    ...,                                          # <- nn.Conv2d(8, 16, kernel_size=3, padding=1)
    nn.ReLU(),
    ...,                                          # <- nn.MaxPool2d(2)
    nn.Flatten(),
    ...,                                          # <- nn.Linear(16 * 4 * 4, 3)
)
n_cnn = core.count_parameters(cnn)
# ------------------------------------------------------------------------------

In [ ]:
print(cnn)
print("\nparameters:", n_cnn)
for name, p in cnn.named_parameters():
    print(f"  {name:14s} {str(tuple(p.shape)):>16s}  {p.numel():6d}")

with torch.no_grad():
    probe = cnn(torch.tensor(X_train[:4]))
print("\nshape check:", tuple(torch.tensor(X_train[:4]).shape), "->", tuple(probe.shape))

**What you should see.** The layer stack, then `parameters: 2019`, then the
per-tensor breakdown, then `shape check: (4, 1, 16, 16) -> (4, 3)`.

The breakdown is worth a look:

* first convolution: $3 \times 3 \times 1 \times 8 = 72$ weights, 8 biases — **80**
* second convolution: $3 \times 3 \times 8 \times 16 = 1152$ weights, 16 biases — **1168**
* classifier: $256 \times 3 = 768$ weights, 3 biases — **771**

The two convolution layers together hold 1248 parameters and do all the seeing.
The single dense layer at the end holds 771 and does nothing but weigh up what
the convolutions found. In a large network that ratio becomes extreme, and it is
why "the network" and "the feature extractor" are often used as synonyms.

Run the shape check before training, every time. It costs one line and it
catches every flatten-size error, which are otherwise diagnosed by a stack trace
sixty lines deep inside `torch.nn.functional.linear`.

---

## 5 · Training

Full batch, Adam, cross entropy, 300 epochs. Four hundred and twenty images fit
in memory hundreds of times over, so there is no reason to complicate the loop
with mini-batches here. In L6 you will meet the reason mini-batches exist, and it
is not memory.

Record the training loss and the held-out accuracy as you go, so you can plot
both afterwards.

### Your turn

In [ ]:
# TODO 5 --- train the network -----------------------------------------------------------
# Three `...` to replace, in order:
#   line 1  ->  nn.CrossEntropyLoss()                targets are class indices 0, 1, 2
#   line 2  ->  torch.optim.Adam(cnn.parameters(), lr=0.01)
#   line 3  ->  loss_fn(cnn(Xt), yt)                 forward pass and loss
loss_fn   = ...                                   # <- nn.CrossEntropyLoss()
optimiser = ...                                   # <- torch.optim.Adam(cnn.parameters(), lr=0.01)

Xt = torch.tensor(X_train);  yt = torch.tensor(y_train)
Xv = torch.tensor(X_test);   yv = torch.tensor(y_test)

losses, accs = [], []
for epoch in range(300):
    optimiser.zero_grad()
    loss = ...                                    # <- loss_fn(cnn(Xt), yt)
    loss.backward()
    optimiser.step()
    losses.append(float(loss.item()))
    if epoch % 10 == 0:
        with torch.no_grad():
            accs.append((epoch, float((cnn(Xv).argmax(1) == yv).float().mean())))
# ------------------------------------------------------------------------------

In [ ]:
with torch.no_grad():
    pred_train = cnn(Xt).numpy().argmax(axis=1)
    pred_test = cnn(Xv).numpy().argmax(axis=1)

acc_train = float((pred_train == y_train).mean())
acc_test = float((pred_test == y_test).mean())

print(f"final training loss : {losses[-1]:.5f}")
print(f"training accuracy   : {acc_train:.3f}")
print(f"held-out accuracy   : {acc_test:.3f}")
print()
core.print_confusion(core.confusion(y_test, pred_test))

fig, ax = plt.subplots(1, 2, figsize=(11.0, 3.8))
ax[0].plot(losses, lw=1.5, color="#1f77b4")
ax[0].set_yscale("log"); ax[0].set_xlabel("epoch")
ax[0].set_ylabel("cross entropy"); ax[0].set_title("Training loss")
ax[0].grid(alpha=0.25, which="both")
ax[1].plot([a[0] for a in accs], [a[1] for a in accs], "o-", lw=1.5,
           ms=4, color="#0f9d58")
ax[1].set_ylim(0.2, 1.02); ax[1].set_xlabel("epoch")
ax[1].set_ylabel("accuracy"); ax[1].set_title("Held-out accuracy")
ax[1].grid(alpha=0.25)
plt.show()

**What you should see.** A training loss falling from about 1.10 — which is
$\ln 3$, the loss of a model that has learned nothing and guesses uniformly among
three classes — down to somewhere around 0.02, a training accuracy of about
0.99, and a **held-out accuracy of about 0.98**. The confusion matrix should be
nearly diagonal, with any mistakes concentrated on shallow cracks read as clean
plate.

The exact figures move by a percentage point or two with your PyTorch version,
because the initialisation is random. If yours is within a couple of points of
0.98 you have it right.

Notice where the loss starts. $\ln 3 = 1.0986$ is not a coincidence and it is
worth memorising: a three-class cross entropy that begins anywhere else means
your labels, your shapes or your initialisation are wrong. The equivalent
numbers are $\ln 2 = 0.693$ for two classes and $\ln 10 = 2.303$ for ten.

---

## 6 · What the first layer learned

The eight kernels of the first convolution are now numbers found by gradient
descent rather than chosen by you. Look at them, and at what they produce.

In [ ]:
W = cnn[0].weight.detach().numpy()          # (8, 1, 3, 3)
print("first-layer kernels:", W.shape)

fig, axes = plt.subplots(1, 8, figsize=(13.0, 2.0))
for i in range(8):
    core.show_kernel(W[i, 0], ax=axes[i], title=f"k{i}")
fig.suptitle("The eight learned 3 x 3 kernels", fontsize=12)
fig.tight_layout()
plt.show()

crack = X_test[y_test == 1][0]
with torch.no_grad():
    maps = cnn[0](torch.tensor(crack[None])).numpy()[0]
core.plot_feature_maps(maps, image=crack[0],
                       title="First-layer feature maps for one cracked plate")
plt.show()

**What you should see.** Eight small red-and-blue squares, and then a row of
nine pictures: the input crack, and the eight feature maps it produces.

Do not expect the learned kernels to look like the textbook Sobel operators from
section 2. Some will; most will look like noisy versions of an edge or blob
detector, and one or two will look like nothing at all. Three reasons, all of
them worth knowing.

**The network has no reason to be tidy.** Nothing in the loss rewards
interpretable kernels. It rewards a low cross entropy, and a messy basis that
spans the right space does that just as well as a clean one.

**Some units are dead.** A ReLU unit whose input is negative for every training
image contributes nothing and receives no gradient, so its kernel stays close to
its initial random values forever. In a layer of eight, one or two dead units is
normal. You saw the same phenomenon in Ex_04's width sweep.

**Eight kernels are more than this task needs.** Three classes distinguished by
"line", "blob" or "neither" do not require eight distinct detectors, so several
are redundant.

The feature maps are easier to read than the kernels. Look for a map in which the
crack stands out as a bright or dark streak against a flat background: that is a
channel which has specialised, and the flatness elsewhere is the point — it is
the layer saying "nothing here".

---

## 7 · The comparison that decides the argument

Everything so far has been an argument from parameter counts. Parameter counts
are not accuracy. Train a dense network on the same images, with **eight times
as many parameters**, and see which wins.

The dense network flattens the 16 by 16 image into a 256-vector and puts one
hidden layer of 64 units on it: $256 \times 64 + 64 + 64 \times 3 + 3 = 16{,}643$
parameters against the convolutional network's 2019.

### Your turn

In [ ]:
# TODO 6 --- the dense baseline, same recipe ----------------------------------------------
# Two `...` to replace:
#   line 1  ->  nn.Linear(16 * 16, 64)             every pixel to 64 hidden units
#   line 2  ->  loss_fn(mlp(Xt), yt)               the same loss, this model
core.set_seed(0)
mlp = nn.Sequential(nn.Flatten(),
                    ..., nn.ReLU(),               # <- nn.Linear(16 * 16, 64)
                    nn.Linear(64, 3))
optimiser_mlp = torch.optim.Adam(mlp.parameters(), lr=0.01)
losses_mlp = []
for epoch in range(300):
    optimiser_mlp.zero_grad()
    loss = ...                                    # <- loss_fn(mlp(Xt), yt)
    loss.backward()
    optimiser_mlp.step()
    losses_mlp.append(float(loss.item()))

with torch.no_grad():
    acc_train_mlp = float((mlp(Xt).argmax(1).numpy() == y_train).mean())
    acc_test_mlp  = float((mlp(Xv).argmax(1).numpy() == y_test).mean())
# ------------------------------------------------------------------------------

In [ ]:
n_mlp = core.count_parameters(mlp)
rows = [["convolutional", f"{n_cnn:,}", f"{acc_train:.3f}", f"{acc_test:.3f}",
         f"{acc_train - acc_test:+.3f}"],
        ["dense", f"{n_mlp:,}", f"{acc_train_mlp:.3f}", f"{acc_test_mlp:.3f}",
         f"{acc_train_mlp - acc_test_mlp:+.3f}"]]
print(core.error_table(rows, ["model", "parameters", "train acc",
                              "held-out acc", "gap"]))

fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.plot(losses, lw=1.6, color="#1f77b4", label=f"convolutional ({n_cnn:,} par.)")
ax.plot(losses_mlp, lw=1.6, color="#d94f2b", label=f"dense ({n_mlp:,} par.)")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("cross entropy")
ax.set_title("Same data, same optimiser, same epochs")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

**What you should see.** Something close to

| model | parameters | train acc | held-out acc | gap |
| --- | --- | --- | --- | --- |
| convolutional | 2,019 | 0.995 | 0.983 | +0.012 |
| dense | 16,643 | 0.888 | 0.722 | +0.166 |

The dense network has eight times the parameters and is worse on the training
set *and* far worse on the held-out set. Both halves of that sentence deserve
attention.

**Worse on the held-out set** is the expected result: 16,643 parameters on 420
images is a recipe for memorisation, and the gap of about 0.17 between training
and held-out accuracy is that memorisation being measured. This is the
overfitting picture from L4.2, arriving here through architecture rather than
through model size.

**Worse on the training set too** is the more interesting result, and it is not
overfitting. The dense network has ample capacity to fit 420 images perfectly;
it simply has not, in 300 epochs, from this initialisation. It has to discover
from data that a crack at the top left and a crack at the bottom right are the
same thing, and the only way it can is to learn near-identical weights in every
region of the image, separately, from the handful of examples that happen to put
a defect in each. The convolution was **told** this, for free, by having one
kernel used everywhere.

That is what an architectural prior is: not a restriction that costs you
accuracy, but knowledge you did not have to buy with data. In L5.1's phrasing,
weight sharing is what makes the local rule *learnable* rather than merely
*representable*.

One honest qualification. This comparison is deliberately favourable: the defect
is uniformly distributed over the plate, so translation invariance is exactly
true. Give the dense network ten thousand images instead of six hundred and the
gap narrows. Give it a task where position genuinely matters, and it wins.
Report the conditions, not just the number.

---

## 8 · Save

Notebook 05 reads this file.

In [ ]:
import os
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb01_cnn.npz")
np.savez(path,
         n_cnn=n_cnn, n_mlp=n_mlp,
         acc_train=acc_train, acc_test=acc_test,
         acc_train_mlp=acc_train_mlp, acc_test_mlp=acc_test_mlp,
         dense_16=dense_16, conv_16=conv_16,
         losses=np.asarray(losses), losses_mlp=np.asarray(losses_mlp),
         confusion=core.confusion(y_test, pred_test))
print("wrote", path)

**What you should see.** `wrote .../Ex05_outputs/nb01_cnn.npz`.

---

## 9 · Before you move on

Answer these here.

1. The convolution's parameter count contains no image size. Name one situation
   in engineering imaging where that is exactly what you want, and one where it
   throws away information you needed.
2. You padded with zeros so that the output stayed 16 by 16. What does a zero
   pixel mean, physically, on the border of a radiograph — and what would a
   better choice of padding be for this particular image?
3. Pooling turned equivariance into invariance. If you wanted to report *where*
   the crack is, and not only that there is one, which part of this architecture
   would you have to change?
4. The dense network was worse on the **training** set. Explain that to a
   colleague who says "more parameters means more capacity means a better fit".

---

Continue with **`Ex05_02_graph_basics.ipynb`**, which takes the same idea — a
local rule applied everywhere — off the grid.

*Write your answers here.*

1.
2.
3.
4.